# NextRun — game video → points, rebounds, assists (Colab)

One pipeline, three models:
- **Players**: YOLO11-pose + BoT-SORT tracking (who is where, hands for possession, feet for court position)
- **Ball + rim**: `models/ball_hoop_model.pt` (shots, makes, rebounds)
- **Court**: `models/court_model.pt` (24 court keypoints → court position in feet → 2PT / 3PT)

→ possession, passes, shots / makes / 3PT, rebounds, assists → box score (PTS, REB, AST first).

**Runtime → Change runtime type → GPU** before running. Stages write into `OUT`, so you can re-run
analysis (new tags / thresholds) without re-detecting.

In [ ]:
!git clone -b model_merge https://github.com/aisutd/nextrun.git /content/nextrun || (cd /content/nextrun && git pull)
%cd /content/nextrun
!pip install -q -e ".[detect]"

## 1. Video
Use your own fixed-camera game (upload or Google Drive), or the TrackID3x3 3x3 sample.

In [ ]:
import os
# Option A: Google Drive
# from google.colab import drive; drive.mount('/content/drive'); VIDEO = '/content/drive/MyDrive/games/game1.mp4'
# Option B: TrackID3x3 sample (1.2 GB) -> first 60 s
if not os.path.exists('/content/sample.mp4'):
    !gdown 13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M -O /content/sample_full.mp4
    !ffmpeg -y -loglevel error -i /content/sample_full.mp4 -t 60 -c copy /content/sample.mp4
VIDEO = '/content/sample.mp4'
OUT = '/content/out/game1'

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
from nextrun_cv import Config, CourtSpec, pipeline
from nextrun_cv.video import probe

cfg = Config()
cfg.court = CourtSpec.preset('highschool')   # highschool | fiba | nba | ncaa  (sets the 3PT line)
cfg.stride = 1                          # 2 = half the frames (faster, worse for dribbles)
cfg.max_frames = None                   # e.g. 600 for a quick test
print(probe(VIDEO))

## 2. Detect + track (GPU-heavy, run once)

In [ ]:
frames = pipeline.detect(VIDEO, OUT, cfg)

## 3. Tag who is playing
The frame below shows tracker IDs. Fill in `TAGS` (track id → name + team). Untagged tracks are
re-linked to tagged players by jersey colour + continuity, or auto-assigned a team.

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT}/tagging_frame.jpg', width=960))

In [ ]:
import json
TAGS = {
    # "1": {"name": "Tushaar", "team": "Red"},
    # "4": {"name": "Juan",    "team": "Blue"},
}
json.dump(TAGS, open(f'{OUT}/tags.json', 'w'), indent=2)

## 4. Court calibration
`auto` tries the court-keypoint model (`models/court_model.pt`: high-school gyms, corner + sideline cameras),
then line registration. If the overlay looks wrong, click landmarks manually (most reliable for a fixed
camera — do it once per camera setup).

In [ ]:
import cv2
from nextrun_cv.viz.overlay import draw_court
from nextrun_cv.video import read_frame
frame0 = read_frame(VIDEO, frames[0].idx)
try:
    calib = pipeline.calibrate(VIDEO, OUT, cfg)
    cv2.imwrite('/content/court_check.jpg', draw_court(frame0, calib))
    display(Image('/content/court_check.jpg', width=960))
except Exception as e:
    print('auto calibration failed:', e)

In [ ]:
# Manual fallback: shows the landmark map, then click each landmark on the frame (Skip if not visible)
from nextrun_cv.court.manual import colab_click_landmarks, landmark_reference_image, DEFAULT_HALF_COURT_POINTS
cv2.imwrite('/content/landmarks.jpg', landmark_reference_image(cfg.court))
display(Image('/content/landmarks.jpg', width=900))
POINTS = colab_click_landmarks(frame0, DEFAULT_HALF_COURT_POINTS)
json.dump(POINTS, open(f'{OUT}/court_points.json', 'w'))
cfg.court_method = 'manual'
calib = pipeline.calibrate(VIDEO, OUT, cfg, manual_points=POINTS)
cv2.imwrite('/content/court_check.jpg', draw_court(frame0, calib))
display(Image('/content/court_check.jpg', width=960))

## 5. Analyze → box score
`PTS`, `REB`, `AST` are the headline stats. Shots with `three_close=True` landed within 1.5 ft of the
3-point line (about the court-mapping error), so double-check those.

In [ ]:
import pandas as pd
analysis, ids, frames_a = pipeline.analyze(OUT, cfg)
display(pd.read_csv(f'{OUT}/boxscore.csv'))
display(pd.read_csv(f'{OUT}/team_totals.csv'))
for name in ['shot_chart.png', 'pass_network.png', 'possession.png']:
    if os.path.exists(f'{OUT}/{name}'):
        display(Image(f'{OUT}/{name}', width=900))

In [ ]:
# event log
pd.DataFrame([{'t': round(e.t, 2), 'type': e.type, 'player': e.player, 'team': e.team,
               **{k: v for k, v in e.data.items() if k in ('made', 'three', 'three_close', 'dunk', 'to', 'offensive', 'distance_m')}}
              for e in analysis.events if e.type != 'dribble'])

## 6. Annotated video

In [ ]:
path = pipeline.render(VIDEO, OUT, cfg, analysis, ids, frames_a)
!ffmpeg -y -loglevel error -i {path} -vcodec libx264 -pix_fmt yuv420p /content/annotated_h264.mp4
from google.colab import files
files.download('/content/annotated_h264.mp4')

## 7. Tune / evaluate
Thresholds live in `cfg.possession`, `cfg.events`, `cfg.ball` — change and re-run step 5 (no re-detection).
Hand-label a clip (`src/scripts/score_events.py` docstring shows the format) and score:
```
!python src/scripts/score_events.py {OUT}/events.json labels/game1.json
```